# LPD-28867

This notebook walks through the process of injecting recovered data into BigQuery. Step to complete a successful recovery includes:
* Load BiqQuery `events`, `eventProperites` and `session` data
* Load JSON parsed recovered `events`, `eventProperties` and `sessions` data
* Validate data
* Update recovered events with relevant session id data
* Store events into a _temporary_ BigQuery table with the same schema as the target one
* Take a snapshot of the existing table
* Execute a BigQuery `MERGE` statement to inject new data

Eventually update all the views and tables that have a dependency with the newly injected event and session data

---

In [ ]:
# Imports

import os
import sys
import tempfile
import time
from typing import List

import pyspark.sql.functions as F

from google.cloud import bigquery

from pyspark import SparkConf
from pyspark.sql import DataFrame, SparkSession
from pyspark.sql.window import Window


In [ ]:
# Spark management
if os.environ.get('PYTHONPATH') is None:
    os.environ['PYTHONPATH']=''

def get_spark_conf(application_name: str, project_id: str, region: str, materialization_dataset: str):
    spark_conf=SparkConf() #get_spark_conf(application_name=app_name, master='yarn')

    spark_conf.setAppName(application_name)
    spark_conf.setMaster('yarn')

    spark_conf.set('spark.jars', 'gs://spark-lib/bigquery/spark-bigquery-latest_2.12.jar')
    spark_conf.set('spark.sql.shuffle.partitions', 128)

    spark_conf.set('materializationDataset', materialization_dataset)
    spark_conf.set('viewsEnabled', 'true')
    spark_conf.set('temporaryGcsBucket', f'{project_id}-dataproc-{region}/temp_write_bucket_for_bq_writes')

    spark_conf.set('spark.serializer', 'org.apache.spark.serializer.KryoSerializer')
    spark_conf.set('spark.kryoserializer.buffer.max', '2000m')

    spark_conf.set('spark.driver.maxResultSize', '0')

    spark_conf.set('spark.executor.cores', '2')
    spark_conf.set('spark.executor.memory', '9g')
    
    return spark_conf

def get_spark_session(application_name=None, spark_conf=None):
    conf_args = '--conf spark.driver.extraJavaOptions="-Dio.netty.tryReflectionSetAccessible=true -XX:ThreadStackSize=8192"'
    # https://github.com/apache/arrow/pull/4522

    jar_jargs = '--jars=gs://spark-lib/bigquery/spark-bigquery-latest_2.12.jar'

    command = 'pyspark-shell'

    os.environ['PYSPARK_SUBMIT_ARGS'] = " ".join([conf_args, jar_jargs, command])

    spark_session_builder = SparkSession.Builder()

    spark_session_builder = spark_session_builder.config(conf=spark_conf or get_spark_conf(application_name))

    spark = spark_session_builder.getOrCreate()
    
    return spark

def create_materialization_dataset(dataset_name: str, project_id: str, region: str) -> str:
    client = bigquery.Client()

    dataset_id = "{}.{}".format(project_id, dataset_name)

    dataset = None
    
    try:
        dataset = client.get_dataset(dataset_id)
    except Exception:
        pass

    if dataset is None:
        # Construct a full Dataset object to send to the API.
        dataset = bigquery.Dataset(dataset_id)

        dataset.location = region

        # Send the dataset to the API for creation, with an explicit timeout.
        # Raises google.api_core.exceptions.Conflict if the Dataset already
        # exists within the project.

        dataset = client.create_dataset(dataset, timeout=30)  # Make an API request.

        print("Created dataset {}.{}".format(client.project, dataset.dataset_id))

    return dataset.dataset_id

### GCS

def load_from_gcs(path: str, filter_condition: str = None, schema = None) -> DataFrame:
    tmp_file = os.path.join(tempfile.mkdtemp(), 'gcs_file_list.txt')
    
    # jupyter notebook trick
    !gsutil ls -r $path > $tmp_file
    
    with open(tmp_file, 'r') as tmp_file_handler:
        path_list = tmp_file_handler.read().split("\n")[:-1]

    df = spark.read.json(
        path=path_list,
        schema=schema
    )
    
    if filter_condition is not None:
        df = df.filter(filter_condition)
    
    return df.cache()

### BigQuery

def load_bq_events(project_id: str, event_date: str):
    return spark.read.format(
            'bigquery'
        ).load(
            f"""
            SELECT
              *
            FROM
              {project_id}.event
            WHERE
                DATE(eventDate) = '{event_date}'
            """
        )

def load_bq_event_properties(project_id: str, event_date: str):
    return spark.read.format(
            'bigquery'
        ).load(
            f"""
            SELECT
              *
            FROM
              {project_id}.eventproperty
            WHERE
                DATE(eventDate) = '{event_date}'
            """
        )

def load_bq_sessions(project_id: str, session_start: str):
    return spark.read.format(
        'bigquery'
    ).load(
        f"""
            SELECT
              *
            FROM
              `{project_id}.session`
            WHERE
              DATE(sessionStart) = '{session_start}'
        """
    )

def load_from_bigquery(table_name: str, projects: List[str], date_str: str) -> DataFrame:
    if table_name == 'event':
        _callable = load_bq_events
    elif table_name == 'eventproperty':
        _callable = load_bq_event_properties
    elif table_name == 'session':
        _callable = load_bq_sessions
    else:
        raise RuntimeError('Unexpected table name')
        
    df = _callable(projects[0], date_str)
    
    error_count = 0

    for project in projects[1:]:
        try:
            df = df.unionByName(_callable(project, date_str), allowMissingColumns=True)
        except Exception as e:
            print(f'Error fetching: {project} - {str(e)}')
            error_count += 1

    print(f"Feched {len(projects) - error_count}, {error_count} errored when fetching")    

    return df.cache()

### Utils

def get_column_split(gcs_dataframe: DataFrame, bq_dataframe: DataFrame, initial_columns: List[str]):
    all_columns = gcs_dataframe.columns
    join_columns = initial_columns
    skip_columns = []
    
    join_count = gcs_dataframe.selectExpr(
        join_columns + ['true AS gcs_event']
    ).join(
        bq_dataframe.selectExpr(
            join_columns + ['true AS bq_event']
        ),
        on=join_columns,
        how='full'
    ).count()

    for col in [c for c in all_columns if c not in join_columns]:    
        test_join = join_columns + [col]

        test_join_count = gcs_dataframe.selectExpr(
            test_join + ['true AS gcs_event']
        ).join(
            bq_dataframe.selectExpr(
                test_join + ['true AS bq_event']
            ),
            on=test_join,
            how='full'
        ).count()

        if test_join_count != join_count:
            skip_columns += [col]
            
            continue

        join_columns += [col]
    
    return join_columns, skip_columns

def compute_overlapping_session(df: DataFrame, partition_columns: List[str]) -> DataFrame:
    """
        Overlap Cases:
        
        * overlap_a:
        
        prev_session: --###########-------
        session:      -----###########----

        * overlap_b:
        
        prev_session: --################--
        session:      -----###########----

        * overlap_c:
        
        prev_session: --------######------
        session:      -----###########----

        * overlap_d:
        
        prev_session: --------##########--
        session:      -----###########----

    """
    lag_w = Window.partitionBy(partition_columns).orderBy(F.col('sessionStart_unix'))
    
    return df.withColumn(
       'sessionStart_unix',
        F.col('sessionStart').cast("double")
    ).withColumn(
       'sessionEnd_unix',
        F.col('sessionEnd').cast("double")
    ).withColumn(
       'previous_sessionStart_unix',
        F.lag('sessionStart_unix', 1).over(lag_w)
    ).withColumn(
       'previous_sessionEnd_unix',
        F.lag('sessionEnd_unix', 1).over(lag_w)
    ).withColumn(
        'row_num',
        F.row_number().over(lag_w)
    ).withColumn(
        'overlap_a',
        F.when(
            (F.col('previous_sessionEnd_unix') >= F.col('sessionStart_unix')) & (F.col('previous_sessionEnd_unix') <= F.col('sessionEnd_unix')),
            1
        ).otherwise(
            0
        )
    ).withColumn(
        'overlap_b',
        F.when(
            (F.col('previous_sessionStart_unix') <= F.col('sessionStart_unix')) & (F.col('previous_sessionEnd_unix') >= F.col('sessionEnd_unix')),
            1
        ).otherwise(
            0
        )
    ).withColumn(
        'overlap_c',
        F.when(
            (F.col('previous_sessionStart_unix') >= F.col('sessionStart_unix')) & (F.col('previous_sessionEnd_unix') <= F.col('sessionEnd_unix')),
            1
        ).otherwise(
            0
        )
    ).withColumn(
        'overlap_d',
        F.when(
            (F.col('previous_sessionStart_unix') >= F.col('sessionStart_unix')) & (F.col('previous_sessionStart_unix') <= F.col('sessionEnd_unix')),
            1
        ).otherwise(
            0
        )
    ).withColumn(
       'overlap',
        F.when(
            F.expr('(overlap_a + overlap_b + overlap_c + overlap_d) > 0'),
            True
        ).otherwise(
            False
        )
    )

def merge_and_rank_sessions_by_duration(df: DataFrame, partition_columns: List[str]) -> DataFrame:
    session_num_w = Window.partitionBy(partition_columns).orderBy(['sessionStart_unix', 'row_num'])
    
    session_row_num_w = Window.partitionBy(partition_columns + ['session_num']).orderBy(F.col('duration').desc())

    session_new_w = Window.partitionBy(partition_columns + ['session_num'])

    return df.withColumn(
            "session_num",
            F.when(
                F.col('overlap') == True,
                F.lag(F.col('row_num'), 1).over(session_num_w)
            ).otherwise(
                F.col('row_num')
            )
        ).withColumn(
            'session_row_num',
            F.row_number().over(session_row_num_w)
        ).withColumn(
            "sessionStart_new",
            F.min('sessionStart').over(session_new_w.orderBy(F.col('sessionStart')))
        ).withColumn(
            "sessionEnd_new",
            F.max('sessionEnd').over(session_new_w.orderBy(F.col('sessionEnd')))
        )

def run_bigquery_query(sql: str, blocking=True):
    client = bigquery.Client()

    response = client.query(sql)
    
    if blocking:
        while not response.done():
            time.sleep(1.0)
    
    return response
        
def create_bigquery_table_from_existing_schema(source_dataset: str, destination_dataset: str, table_name: str, blocking=True):
    sql = f"""
        CREATE TABLE IF NOT EXISTS {destination_dataset}.{source_dataset}_{table_name} AS
        SELECT *
        FROM {source_dataset}.{table_name}
        LIMIT 0
    """

    return run_bigquery_query(sql)

def create_table_snapshot(source_dataset: str, source_table: str, snapshot_dataset: str, snapshot_name: str, blocking=True):
    sql = f"""
        CREATE SNAPSHOT TABLE {PROJECT_ID}.{snapshot_dataset}.{snapshot_name}
        CLONE {PROJECT_ID}.{source_dataset}.{source_table};
    """
    
    return run_bigquery_query(sql)

def delete_bigquery_snapshot(snapshot_dataset: str, snapshot_name: str, blocking=True):
    sql = f"""
        DROP SNAPSHOT TABLE {PROJECT_ID}.{snapshot_dataset}.{snapshot_name}
    """

    return run_bigquery_query(sql)

### Environment setup

Configure `PROJECT_ID`, `REGION` and all the parameter to specilize the notebook to recover a single Analytics Cloud workspace

In [ ]:
PROJECT_ID = !gcloud config get project
PROJECT_ID = PROJECT_ID[0]

REGION = 'europe-west3'

# --- Issue specific

APPLICATION_NAME = 'LPD-28867'
EVENT_DATE = '2024-05-17'
WORKSPACE_ID = 'asah9e448c7d88364ecfbd83cf7728bd5bc5' # 'asah087a17be3b864acc8cece5bf30319d30'

print(f"Working on Google project: {PROJECT_ID} in region: {REGION}. Recovering events for {WORKSPACE_ID} on date '{EVENT_DATE}'")

In [ ]:
spark = get_spark_session(
    application_name=APPLICATION_NAME,
    spark_conf=get_spark_conf(
        application_name=APPLICATION_NAME,
        project_id=PROJECT_ID,
        region=REGION,
        materialization_dataset=create_materialization_dataset(
            dataset_name=APPLICATION_NAME.replace('-','_'),
            project_id=PROJECT_ID,
            region=REGION
        )
    )
)

spark

---

### Load Existing Data

#### Load BigQuery data

In [ ]:
bq_events = load_from_bigquery(
    table_name='event',
    projects=[WORKSPACE_ID],
    date_str=EVENT_DATE
)

print(f"Loaded {bq_events.count()} events from BigQuery")

In [ ]:
bq_event_properties = load_from_bigquery(
    table_name='eventproperty',
    projects=[WORKSPACE_ID],
    date_str=EVENT_DATE
)

print(f"Loaded {bq_event_properties.count()} event properties from BigQuery")

In [ ]:
bq_sessions = load_from_bigquery(
    table_name='session',
    projects=[WORKSPACE_ID],
    date_str=EVENT_DATE
)

print(f"Loaded {bq_sessions.count()} sessions from BigQuery")

---

#### Load GCS data

When loading data we also drop duplicate records as this may be a by-product of the data recovery process

In [ ]:
gcs_events = load_from_gcs(
    path=f"gs://{PROJECT_ID}-dataflow/batch/output/{EVENT_DATE.replace('-', '')}/events/*.jsonl",
    filter_condition=f'projectId = "{WORKSPACE_ID}"',
    schema=bq_events.schema
).dropDuplicates()

print(f"Loaded {gcs_events.count()} events from GCS")

In [ ]:
gcs_event_properties = load_from_gcs(
    path=f"gs://{PROJECT_ID}-dataflow/batch/output/{EVENT_DATE.replace('-', '')}/eventproperties/*.jsonl",
    filter_condition=f'projectId = "{WORKSPACE_ID}"',
    schema=bq_event_properties.schema
).dropDuplicates()

print(f"Loaded {gcs_event_properties.count()} event properties from GCS")

In [ ]:
gcs_sessions = load_from_gcs(
    path=f"gs://{PROJECT_ID}-dataflow/batch/output/{EVENT_DATE.replace('-', '')}/sessions/*.jsonl",
    filter_condition=f'projectId = "{WORKSPACE_ID}"',
    schema=bq_sessions.schema
).dropDuplicates()

print(f"Loaded {gcs_sessions.count()} sessions from GCS")

---

### Validate data

#### Validate Events data

Cross checkin:
* How many events are in BigQuery that are not available on GCS 
* How many events are in GCS that are not available on BigQuery 

In [ ]:
common_columns = list(set(bq_events.columns) & set(gcs_events.columns))

In [ ]:
print(f"GCS events count: {gcs_events.count()}, BigQuery events count: {bq_events.count()}. Delta: {gcs_events.count() - bq_events.count()}")

Delta count includes events that are available in BigQuery and not on GCS however this does not accout for events that are not available in BigQuery but available on GCS

In [ ]:
non_matching_gcs_events = gcs_events.exceptAll(bq_events)

non_matching_bq_events = bq_events.exceptAll(gcs_events)

print(f"Non matching GCS events: {non_matching_gcs_events.count()}. Matching events: {gcs_events.count() - non_matching_gcs_events.count()}")
print(f"Non matching BQ events: {non_matching_bq_events.count()}. Matching events: {bq_events.count() - non_matching_bq_events.count()}")

Non matching GCS events are events that are different - by any field - from the events stored in BigQuery. Those are the events that we will recoved and insert back into BigQuery

Non matching BigQuery events are events that are different - by any fields - from the events stored on GCS. This is a concerning aspect. Those events are available in bigquery but are not found on GCS, likely the backup pipeline was affected by the outage too.

*Unfortunately:* We can not use this method since events with not matching `sessionId` would end up being labeled as mismatching

In [ ]:
non_matching_gcs_events = gcs_events.drop('sessionId').exceptAll(bq_events.drop('sessionId'))

non_matching_bq_events = bq_events.drop('sessionId').exceptAll(gcs_events.drop('sessionId'))

print(f"Non matching GCS events (excluding sessionId): {non_matching_gcs_events.count()}. Matching events: {gcs_events.count() - non_matching_gcs_events.count()}")
print(f"Non matching BQ events (excluding sessionId): {non_matching_bq_events.count()}. Matching events: {bq_events.count() - non_matching_bq_events.count()}")

This second round dropping the `sessionIs` column shows different numbers. This enphasize the differences in terms of **session attribution**.

What are the mismatching columns?
The following function returns 2 lists. One containes the list of columns that do not change the join line count, the second one is the list of columns that alter the column count starting from a very basic column set.

What does it mean?
Doing a full_outer join when column values mismatch leads to duplicated rows increasing the row count.

#### Validate Session data

Cross checkin:
* How many sessions are in BigQuery that are not available on GCS 
* How many sessions are in GCS that are not available on BigQuery 

In [ ]:
non_matching_gcs_sessions = gcs_sessions.exceptAll(bq_sessions)
non_matching_bq_sessions = bq_sessions.exceptAll(gcs_sessions)

In [ ]:
print(f"Non matching GCS sessions: {non_matching_gcs_sessions.count()}. Matching sessions: {gcs_sessions.count() - non_matching_gcs_sessions.count()}")
print(f"Non matching BQ sessions: {non_matching_bq_sessions.count()}. Matching sessions: {bq_sessions.count() - non_matching_bq_sessions.count()}")

The inspection shows us three different scenarios:
* When `sessionStart` matches we should prioritize the longer session, generally coming from GCS
* A session could start earlier and overlap with an existing (BQ) one.
* A session could start earlier and end later than an existing session from BQ

---

### Prepare new data

#### Sessions: Prepare new Session data

In [ ]:
base_session_columns = ['projectId', 'channelId', 'userId']

We now have to account for overlapping sessions with a different `sessionStart`

In [ ]:
all_sessions = compute_overlapping_session(
    gcs_sessions.withColumn(
        'src', F.lit('gcs')
    ).unionByName(
        bq_sessions.withColumn(
            'src', F.lit('bq')
        )
    ), 
    base_session_columns
)

We now add a sort within group indicator `session_row_number` based on the session `duration`. Longer sessions will have a `session_row_number` column set to `1`.

This allow us to filter by `session_row_number == 1` and remove all the sessions with a common `startSession` that are shorter thus missing events.

In [ ]:
all_sessions = merge_and_rank_sessions_by_duration(all_sessions, base_session_columns)

In [ ]:
all_sessions = all_sessions.filter('session_row_num = 1')

In [ ]:
all_sessions.count()

In [ ]:
session_count = 0

while session_count != all_sessions.count():
    session_count = all_sessions.count()
    print(f"Sessions count: {session_count}")
    
    all_sessions = compute_overlapping_session(all_sessions, base_session_columns)
    all_sessions = merge_and_rank_sessions_by_duration(all_sessions, base_session_columns)
    all_sessions = all_sessions.filter('session_row_num = 1')



In [ ]:
all_sessions.groupBy('src').count().show()

In [ ]:
new_sessions = all_sessions.select(gcs_sessions.columns)

The `new_session` DataFrame contains all the new session data. Those are the session that were not match in the validation step and are adjusted to keep the longest session duration in case of overlap.

---

#### Events: Assign new session ids

Assign new `sessionId` values to all GCS events in `gcs_events`

In [ ]:
df = gcs_events.join(
    all_sessions.selectExpr('userId AS userId_session', 'channelId AS channelId_session', 'id AS sessionId_new', 'sessionStart', 'sessionEnd', 'sessionStart_new', 'sessionEnd_new'),
    on=[
        gcs_events['userId'] == F.col('userId_session'), 
        gcs_events['channelId'] == F.col('channelId_session'),
        gcs_events['eventDate'] >= all_sessions['sessionStart_new'], 
        gcs_events['eventDate'] <= all_sessions['sessionEnd_new']
    ],
    how='left'
)

We confirm there are no duplicate event `id` as it would imply join issues.

Some events ot duplicate on the source side, those events have same `id` but different `createDate`

In [ ]:
duplicate_session_count = df.groupBy('id', 'createDate').count().orderBy('count', ascending=False).filter('count > 1').count()

assert(duplicate_session_count == 0)

And confirm no events got dropped in the transaction

In [ ]:
assert(gcs_events.count() == df.count())

In [ ]:
df.filter('sessionId_new is not null').count()

In [ ]:
df.filter('sessionId_new is null').count()

There are some events who did not match with the sessions and got `NULL` as new sessionId. We are going to ignore them as the number is low the to correct the issue we would need to re-compute all the sessions again.

This is because some sessions on BQ would overlap with GCS near the extreme session boundaries.

In [ ]:
df = df.filter('sessionId_new is not null')

In [ ]:
new_session_events_count = df.filter('sessionId_new is not null AND sessionId != sessionId_new').count()

print(f"Number of events that got a new session id: {new_session_events_count}")

In [ ]:
mismatching_session_events_count = df.filter(
    'sessionId_new is not null AND sessionId != sessionId_new'
).select(
    'userId', 'sessionId','sessionId_new','eventDate', 'sessionStart', 'sessionEnd'
).withColumn(
    'check',
    F.when(
        (F.col('eventDate') >= F.col('sessionStart')) & (F.col('eventDate') <= F.col('sessionEnd')),
        True
    ).otherwise(False)
).filter('check is false').orderBy('userId','sessionStart').count()

assert(mismatching_session_events_count == 0)

We also confirm all the events, including the one with a new session are still included in the session boundaries

In [ ]:
new_events = df.drop(
    'sessionId'
).withColumnRenamed(
    'sessionId_new',
    'sessionId'
).select(
    gcs_events.columns
)

---

#### EventProperty: Prepare new event properties

There is little chance we are missing part of the properties. An event was wither processed or not. We are going to:
* Filter the GCS Event ids not avialable in the BQ Event table
* Use the list of ids to filter already existing Event Properties
* Store the new Event Properties into a temporary table
* Merge new properties via BQ `MERGE` statement

In [ ]:
new_event_ids = new_events.select('id').join(
    bq_events.select('id'),
    on=['id'],
    how='leftanti'
)

print(f"Filtered {new_event_ids.count()} ids")

In [ ]:
new_event_properties = gcs_event_properties.select('id').join(
    new_event_ids,
    on=['id'],
    how='inner'
)

print(f"Filtered {new_event_properties.count()} event properties")

---

### Loading New Data into BigQuery

We are going to write data to BigQuery for the final merge. This part will leverage BigQuery capabilities instead of spark itsel:
* Create a temporary dataset.table
* Write from Spark to the dataset nad table created on the previous step
* Take snapshot of the target table
* Execute a `MERGE` statement to merge missing records

#### Events

1. Create temporary table

In [ ]:
response = create_bigquery_table_from_existing_schema(
    source_dataset=WORKSPACE_ID,
    destination_dataset=APPLICATION_NAME.replace('-', '_'),
    table_name='event'
)

print(response)

2. Write data from Spark to BigQuery

In [ ]:
new_events.write.format('bigquery').mode(
    'OVERWRITE'
).option(
    'createDisposition',
    'CREATE_NEVER'
).save(
    f"{APPLICATION_NAME.replace('-','_')}.{WORKSPACE_ID}_event"
)

#### Event Properties

1. Create temporary table

In [ ]:
response = create_bigquery_table_from_existing_schema(
    source_dataset=WORKSPACE_ID,
    destination_dataset=APPLICATION_NAME.replace('-', '_'),
    table_name='eventproperty'
)

print(response)

2. Write data from Spark to BigQuery

In [ ]:
new_event_properties.write.format('bigquery').mode(
    'OVERWRITE'
).option(
    'createDisposition',
    'CREATE_NEVER'
).save(
    f"{APPLICATION_NAME.replace('-','_')}.{WORKSPACE_ID}_eventproperty"
)

#### Sessions

1. Create temporary table

In [ ]:
response = create_bigquery_table_from_existing_schema(
    source_dataset=WORKSPACE_ID,
    destination_dataset=APPLICATION_NAME.replace('-', '_'),
    table_name='session'
)

print(response)

2. Write data from Spark to BigQuery

In [ ]:
new_sessions.write.format('bigquery').mode(
    'OVERWRITE'
).option(
    'createDisposition',
    'CREATE_NEVER'
).save(
    f"{APPLICATION_NAME.replace('-','_')}.{WORKSPACE_ID}_session"
)

---

In [ ]:
raise Exception("STOP HERE")

---

### Snapshot and Merge

We are going to create a table snapshot then execute a merge statement

#### Events

In [ ]:
response = create_table_snapshot(
    source_dataset=APPLICATION_NAME.replace('-','_'),
    source_table=f'{WORKSPACE_ID}_event',
    snapshot_dataset=APPLICATION_NAME.replace('-','_'),
    snapshot_name=f'{WORKSPACE_ID}_event_snapshot'
)

print(f"Snapshot creation for event table completed with status {response.state}")

In [ ]:
merge_events_sql =f"""
    MERGE INTO
        `{PROJECT_ID}.{WORKSPACE_ID}.event` as replica
    USING
        (
            SELECT
                *
            FROM
                `{PROJECT_ID}.{APPLICATION_NAME.replace('-','_')}.{WORKSPACE_ID}_event`
        ) AS staging
    ON (
            staging.projectId = replica.projectId AND
            staging.channelId = replica.channelId AND
            staging.userId = replica.userId AND
            staging.id = replica.id AND
            staging.eventDate = replica.eventDate            
    )
    
    WHEN MATCHED THEN
        UPDATE SET
            replica.sessionId = staging.sessionId
    WHEN NOT MATCHED THEN
        INSERT (
            `applicationId`,
            `browserName`,
            `canonicalUrl`,
            `channelId`,
            `city`,
            `contentLanguageId`,
            `context`,
            `country`,
            `createDate`,
            `dataSourceId`,
            `description`,
            `deviceType`,
            `eventDate`,
            `eventId`,
            `eventProperties`,
            `experienceId`,
            `id`,
            `keywords`,
            `languageId`,
            `platformName`,
            `projectId`,
            `projectTimeZoneId`,
            `referrer`,
            `region`,
            `sessionId`,
            `timezoneOffset`,
            `title`,
            `url`,
            `userId`,
            `variantId`,
            `emailAddressHashed`,
            `assetId`,
            `assetTitle`,
            `experimentId`
        )
        VALUES (
            staging.applicationId,
            staging.browserName,
            staging.canonicalUrl,
            staging.channelId,
            staging.city,
            staging.contentLanguageId,
            staging.context,
            staging.country,
            staging.createDate,
            staging.dataSourceId,
            staging.description,
            staging.deviceType,
            staging.eventDate,
            staging.eventId,
            staging.eventProperties,
            staging.experienceId,
            staging.id,
            staging.keywords,
            staging.languageId,
            staging.platformName,
            staging.projectId,
            staging.projectTimeZoneId,
            staging.referrer,
            staging.region,
            staging.sessionId,
            staging.timezoneOffset,
            staging.title,
            staging.url,
            staging.userId,
            staging.variantId,
            staging.emailAddressHashed,
            staging.assetId,
            staging.assetTitle,
            staging.experimentId
        )
"""

# response = run_bigquery_query(sql=merge_events_sql)

# print(f"Merge event statement completed with status {response.state}")

In [ ]:
# response = delete_bigquery_snapshot(
#     snapshot_dataset=APPLICATION_NAME.replace('-','_'),
#     snapshot_name=f'{WORKSPACE_ID}_event_snapshot'
# )

# print(f"{response.state}")

#### Event Properties

In [ ]:
response = create_table_snapshot(
    source_dataset=APPLICATION_NAME.replace('-','_'),
    source_table=f'{WORKSPACE_ID}_eventproperty',
    snapshot_dataset=APPLICATION_NAME.replace('-','_'),
    snapshot_name=f'{WORKSPACE_ID}_eventproperty_snapshot'
)

print(f"Snapshot creation for eventproperty table completed with status {response.state}")

In [ ]:
merge_eventproperty_sql =f"""
    MERGE INTO
        `{PROJECT_ID}.{WORKSPACE_ID}.eventproperty` as replica
    USING
        (
            SELECT
                *
            FROM
                `{PROJECT_ID}.{APPLICATION_NAME.replace('-','_')}.{WORKSPACE_ID}_eventproperty`
        ) AS staging
    ON (
            staging.projectId = replica.projectId AND
            staging.channelId = replica.channelId AND
            staging.id = replica.id AND
            staging.eventDate = replica.eventDate            
    )
    
    WHEN NOT MATCHED THEN
        INSERT (
            `channelId`,
            `eventDate`,
            `id`,
            `name`,
            `projectId`,
            `value`
        )
        VALUES (
            staging.fullname,
            staging.channelId,
            staging.eventDate,
            staging.id,
            staging.name,
            staging.projectId,
            staging.value
        )
"""

# response = run_bigquery_query(sql=merge_eventproperty_sql)

# print(f"Merge eventproperty statement completed with status {response.state}")

In [ ]:
# response = delete_bigquery_snapshot(
#     snapshot_dataset=APPLICATION_NAME.replace('-','_'),
#     snapshot_name=f'{WORKSPACE_ID}_eventproperty_snapshot'
# )

# print(f"{response.state}")

#### Sessions

In [ ]:
response = create_table_snapshot(
    source_dataset=APPLICATION_NAME.replace('-','_'),
    source_table=f'{WORKSPACE_ID}_session',
    snapshot_dataset=APPLICATION_NAME.replace('-','_'),
    snapshot_name=f'{WORKSPACE_ID}_session_snapshot'
)

print(f"Snapshot creation for session table completed with status {response.state}")

In [ ]:
merge_sessions_sql =f"""
    MERGE INTO
        `{PROJECT_ID}.{WORKSPACE_ID}.session` as target
    USING
        (
            SELECT
                *
            FROM
                `{PROJECT_ID}.{APPLICATION_NAME.replace('-','_')}.{WORKSPACE_ID}_session`
        ) AS source
    ON (
            source.projectId = target.projectId AND
            source.channelId = target.channelId AND
            source.userId = target.userId AND
            source.id = target.id            
    )
    
    WHEN MATCHED
        UPDATE SET
            target.sessionEnd = source.sessionEnd,
            target.sessionStart = source.sessionStart,
            target.acquisitionCampaign = source.acquisitionCampaign,
            target.acquisitionChannel = source.acquisitionChannel,
            target.acquisitionContent = source.acquisitionContent,
            target.acquisitionMedium = source.acquisitionMedium,
            target.acquisitionSource = source.acquisitionSource,
            target.acquisitionTerm = source.acquisitionTerm,
            target.bounce = source.bounce,
            target.browserName = source.browserName,
            target.city = source.city,
            target.country = source.country,
            target.deviceType = source.deviceType,
            target.duration = source.duration,
            target.platformName = source.platformName,
            target.referrers = source.referrers,
            target.region = source.region,
            target.upgraded = source.upgraded,
            target.urls = source.urls
    WHEN NOT MATCHED
        INSERT (
            `channelId`,
            `id`,
            `projectId`,
            `sessionEnd`,
            `sessionStart`,
            `acquisitionCampaign`,
            `acquisitionChannel`,
            `acquisitionContent`,
            `acquisitionMedium`,
            `acquisitionSource`,
            `acquisitionTerm`,
            `bounce`,
            `browserName`,
            `city`,
            `country`,
            `deviceType`,
            `duration`,
            `platformName`,
            `referrers`,
            `region`,
            `userId`,
            `upgraded`,
            `urls`
        )
        VALUES (
            source.channelId,
            source.id,
            source.projectId,
            source.sessionEnd,
            source.sessionStart,
            source.acquisitionCampaign,
            source.acquisitionChannel,
            source.acquisitionContent,
            source.acquisitionMedium,
            source.acquisitionSource,
            source.acquisitionTerm,
            source.bounce,
            source.browserName,
            source.city,
            source.country,
            source.deviceType,
            source.duration,
            source.platformName,
            source.referrers,
            source.region,
            source.userId,
            source.upgraded,
            source.urls
        )

"""

# response = run_bigquery_query(sql=merge_sessions_sql)

# print(f"Merge sessions statement completed with status {response.state}")

In [ ]:
# response = delete_bigquery_snapshot(
#     snapshot_dataset=APPLICATION_NAME.replace('-','_'),
#     snapshot_name=f'{WORKSPACE_ID}_session_snapshot'
# )

# print(f"{response.state}")

---
### Merge metrics jobs

We can rerun merge jobs to align all the views. Starting from [internal page](https://liferay.atlassian.net/wiki/spaces/ENGAC/pages/2546761797/How+to+run+merge+daily+metrics+task+process+for+specific+day) we can adapt those queries to update existing records too

In [ ]:
WORKSPACE_TIME_ZONE_ID = gcs_events.select(
    'projectTimeZoneId'
).distinct(
).rdd.map(lambda r: r[0]).collect()

assert(len(WORKSPACE_TIME_ZONE_ID)=1)

WORKSPACE_TIME_ZONE_ID = WORKSPACE_TIME_ZONE_ID[0]

In [ ]:
WORKSPACE_TIME_ZONE_ID='UTC'

#### Blogs

In [ ]:
blogs_daily_merge_sql = f"""
MERGE INTO
    `{WORKSPACE_ID}.blogdaily` AS replica
USING
    (
        WITH
            BlogEvent AS (
                SELECT
                    Event.assetId,
                    Event.assetTitle,
                    Event.browserName,
                    Event.canonicalUrl,
                    Event.channelId,
                    Event.city,
                    Event.country,
                    Event.deviceType,
                    Event.eventDate,
                    Event.eventId,
                    Event.platformName,
                    Event.region,
                    Event.sessionId,
                    Event.title,
                    Event.userId
                FROM
                    `{WORKSPACE_ID}.event` AS Event
                LEFT JOIN `{WORKSPACE_ID}.eventproperty` AS className ON (
                    DATE(className.eventDate, '{WORKSPACE_TIME_ZONE_ID}') = '{EVENT_DATE}' AND
                    className.id = Event.id AND
                    className.name = 'className' AND
                    className.value = 'com.liferay.blogs.model.BlogsEntry'
                )
                WHERE
                    (
                        (
                            Event.applicationId = 'Blog' AND
                            Event.eventId IN ('blogClicked', 'blogDepthReached', 'blogViewed')
                        ) OR
                        (
                            Event.applicationId = 'Ratings' AND
                            className.value IS NOT NULL
                        )
                    ) AND
                    Event.assetId IS NOT NULL AND
                    Event.assetTitle IS NOT NULL AND
                    Event.canonicalUrl IS NOT NULL AND
                    DATE(Event.eventDate, '{WORKSPACE_TIME_ZONE_ID}') = '{EVENT_DATE}' AND
                    Event.title IS NOT NULL
            ),
            BlogFinalizedEvent AS (
                SELECT
                    BlogEvent.assetId,
                    BlogEvent.assetTitle,
                    BlogEvent.canonicalUrl,
                    BlogEvent.channelId,
                    BlogEvent.eventDate,
                    BlogEvent.eventId,
                    BlogEvent.sessionId,
                    BlogEvent.title,
                    BlogEvent.userId
                FROM
                    BlogEvent
                INNER JOIN `{WORKSPACE_ID}.session` AS Session ON
                    BlogEvent.sessionId = Session.id
                WHERE
                    DATE(Session.sessionStart, '{WORKSPACE_TIME_ZONE_ID}') = '{EVENT_DATE}'
            ),
            CommentEvent AS (
                SELECT
                    Event.assetId,
                    Event.canonicalUrl,
                    Event.channelId,
                    Event.eventDate,
                    Event.title,
                    Event.userId
                FROM
                    `{WORKSPACE_ID}.event` AS Event
                LEFT JOIN `{WORKSPACE_ID}.eventproperty` AS className ON (
                    DATE(className.eventDate, '{WORKSPACE_TIME_ZONE_ID}') = '{EVENT_DATE}' AND
                    className.id = Event.id AND
                    className.name = 'className' AND
                    className.value = 'com.liferay.blogs.model.BlogsEntry'
                )
                WHERE
                    Event.applicationId = 'Comment' AND
                    Event.assetId IS NOT NULL AND
                    Event.canonicalUrl IS NOT NULL AND
                    DATE(Event.eventDate, '{WORKSPACE_TIME_ZONE_ID}') = '{EVENT_DATE}' AND
                    Event.eventId = 'posted' AND
                    Event.title IS NOT NULL
            ),
            BlogComments AS (
                SELECT
                    assetId,
                    canonicalUrl,
                    SUM(1) AS comments,
                    channelId,
                    TIMESTAMP_TRUNC(eventDate, HOUR) AS normalizedEventDate,
                    title AS pageTitle,
                    userId
                FROM
                    CommentEvent
                GROUP BY
                    assetId, canonicalUrl, channelId, normalizedEventDate, title, userId
            ),
            RatingsEvent AS (
                SELECT
                    Event.assetId,
                    Event.canonicalUrl,
                    Event.channelId,
                    Event.eventDate,
                    CAST(score.value AS FLOAT64) AS score,
                    Event.title,
                    Event.userId
                FROM
                    `{WORKSPACE_ID}.event` AS Event
                LEFT JOIN `{WORKSPACE_ID}.eventproperty` AS className ON (
                    DATE(className.eventDate, '{WORKSPACE_TIME_ZONE_ID}') = '{EVENT_DATE}' AND
                    className.id = Event.id AND
                    className.value = 'com.liferay.blogs.model.BlogsEntry' AND
                    className.name = 'className'
                )
                LEFT JOIN `{WORKSPACE_ID}.eventproperty` AS ratingType ON (
                    DATE(ratingType.eventDate, '{WORKSPACE_TIME_ZONE_ID}') = '{EVENT_DATE}' AND
                    ratingType.id = Event.id AND
                    ratingType.value = 'stars' AND
                    ratingType.name = 'ratingType'
                )
                LEFT JOIN `{WORKSPACE_ID}.eventproperty` AS score ON (
                    DATE(score.eventDate, '{WORKSPACE_TIME_ZONE_ID}') = '{EVENT_DATE}' AND
                    score.id = Event.id AND
                    score.name = 'score'
                )
                WHERE
                    Event.applicationId = 'Ratings' AND
                    Event.assetId IS NOT NULL AND
                    Event.canonicalUrl IS NOT NULL AND
                    DATE(Event.eventDate, '{WORKSPACE_TIME_ZONE_ID}') = '{EVENT_DATE}' AND
                    Event.eventId = 'VOTE' AND
                    Event.title IS NOT NULL
            ),
            BlogRatings AS (
                SELECT
                    assetId,
                    canonicalUrl,
                    channelId,
                    TIMESTAMP_TRUNC(eventDate, HOUR) AS normalizedEventDate,
                    title AS pageTitle,
                    SUM(1) AS ratings,
                    score AS ratingsScore,
                    userId
                FROM
                    RatingsEvent AS RatingsEvent1
                WHERE
                    RatingsEvent1.eventDate = (
                        SELECT
                            MAX(RatingsEvent2.eventDate)
                        FROM
                            RatingsEvent RatingsEvent2
                        WHERE
                            RatingsEvent1.assetId = RatingsEvent2.assetId AND
                            RatingsEvent1.userid = RatingsEvent2.userid
                    ) AND score >= 0
                GROUP BY
                    assetId, canonicalUrl, channelId, normalizedEventDate, score,
                    title, userId
            ),
            BlogReadTimes AS (
                SELECT
                    assetId,
                    assetTitle,
                    canonicalUrl,
                    channelId,
                    TIMESTAMP_TRUNC(maxEventDate, HOUR) AS normalizedEventDate,
                    title AS pageTitle,
                    SUM(readTime) AS readTime,
                    userId
                FROM
                    (
                        SELECT
                            assetId,
                            assetTitle,
                            canonicalUrl,
                            channelId,
                            MAX(CASE WHEN eventId != 'blogViewed' THEN eventDate END) AS maxEventDate,
                            (
                                UNIX_SECONDS(MAX(CASE WHEN eventId != 'blogViewed' THEN eventDate END)) -
                                UNIX_SECONDS(MIN(eventDate))
                            ) AS readTime,
                            sessionId,
                            title,
                            userId
                        FROM
                            BlogFinalizedEvent
                        GROUP BY
                            assetId, assetTitle, canonicalUrl, channelId, sessionId,
                            title, userId
                    ) AS TMP
                WHERE
                    maxEventDate IS NOT NULL
                GROUP BY
                    assetId, assetTitle, canonicalUrl, channelId, normalizedEventDate,
                    title, userId
            ),
            BlogViewsAndClicks AS (
                SELECT
                    assetId,
                    assetTitle,
                    browserName,
                    canonicalUrl,
                    channelId,
                    COUNTIF(eventId = 'blogClicked') AS clicks,
                    city,
                    country,
                    TIMESTAMP_TRUNC(eventDate, HOUR) AS normalizedEventDate,
                    deviceType,
                    platformName,
                    region,
                    COUNT(DISTINCT(sessionId)) AS sessions,
                    title AS pageTitle,
                    userId,
                    COUNTIF(eventId = 'blogViewed') AS views
                FROM
                    BlogEvent
                GROUP BY
                    assetId, assetTitle, browserName, canonicalUrl, channelId, city,
                    country, normalizedEventDate, deviceType, platformName,
                    region, title, userId
            ),
            BlogHourly AS (
                SELECT
                    BlogViewsAndClicks.assetId,
                    BlogViewsAndClicks.assetTitle,
                    BlogViewsAndClicks.browserName,
                    BlogViewsAndClicks.canonicalUrl,
                    BlogViewsAndClicks.channelId,
                    BlogViewsAndClicks.city,
                    COALESCE(BlogViewsAndClicks.clicks, 0) AS clicks,
                    COALESCE(BlogComments.comments, 0) AS comments,
                    BlogViewsAndClicks.country,
                    BlogViewsAndClicks.deviceType,
                    BlogViewsAndClicks.normalizedEventDate AS eventDate,
                    BlogViewsAndClicks.pageTitle,
                    BlogViewsAndClicks.platformName,
                    BlogRatings.ratings,
                    BlogRatings.ratingsScore,
                    BlogReadTimes.readTime * 1000 AS readTime,
                    BlogViewsAndClicks.region,
                    BlogViewsAndClicks.sessions,
                    BlogViewsAndClicks.userId,
                    COALESCE(BlogViewsAndClicks.views, 0) AS views
                FROM
                    BlogViewsAndClicks
                LEFT JOIN BlogComments ON (
                    BlogViewsAndClicks.assetId = BlogComments.assetId AND
                    BlogViewsAndClicks.canonicalUrl = BlogComments.canonicalUrl AND
                    BlogViewsAndClicks.channelId = BlogComments.channelId AND
                    BlogViewsAndClicks.normalizedEventDate = BlogComments.normalizedEventDate AND
                    BlogViewsAndClicks.pageTitle = BlogComments.pageTitle AND
                    BlogViewsAndClicks.userId = BlogComments.userId
                )
                LEFT JOIN BlogRatings ON (
                    BlogViewsAndClicks.assetId = BlogRatings.assetId AND
                    BlogViewsAndClicks.canonicalUrl = BlogRatings.canonicalUrl AND
                    BlogViewsAndClicks.channelId = BlogRatings.channelId AND
                    BlogViewsAndClicks.normalizedEventDate = BlogRatings.normalizedEventDate AND
                    BlogViewsAndClicks.pageTitle = BlogRatings.pageTitle AND
                    BlogViewsAndClicks.userId = BlogRatings.userId
                )
                LEFT JOIN BlogReadTimes ON (
                    BlogViewsAndClicks.assetId = BlogReadTimes.assetId AND
                    BlogViewsAndClicks.assetTitle = BlogReadTimes.assetTitle AND
                    BlogViewsAndClicks.canonicalUrl = BlogReadTimes.canonicalUrl AND
                    BlogViewsAndClicks.channelId = BlogReadTimes.channelId AND
                    BlogViewsAndClicks.normalizedEventDate = BlogReadTimes.normalizedEventDate AND
                    BlogViewsAndClicks.pageTitle = BlogReadTimes.pageTitle AND
                    BlogViewsAndClicks.userId = BlogReadTimes.userId
                )
            )
        SELECT
            assetId,
            assetTitle,
            browserName,
            canonicalUrl,
            channelId,
            city,
            SUM(clicks) AS clicks,
            SUM(comments) AS comments,
            country,
            deviceType,
            TIMESTAMP_TRUNC(eventDate, DAY, '{WORKSPACE_TIME_ZONE_ID}') AS eventDate,
            pageTitle,
            platformName,
            SUM(ratings) AS ratings,
            SUM(ratingsScore) AS ratingsScore,
            SUM(readTime) AS readTime,
            region,
            SUM(sessions) AS sessions,
            userId,
            SUM(views) AS views
        FROM
            BlogHourly
        WHERE
            DATE(eventDate, '{WORKSPACE_TIME_ZONE_ID}') = '{EVENT_DATE}'
        GROUP BY
            assetId, assetTitle, browserName, canonicalUrl, channelId, city,
            country, deviceType, eventDate, pageTitle, platformName, region,
            userId
    ) AS staging
ON (
    DATE(replica.eventDate, '{WORKSPACE_TIME_ZONE_ID}') = '{EVENT_DATE}' AND
    staging.assetId = replica.assetId AND
    staging.assetTitle = replica.assetTitle AND
    COALESCE(staging.browserName, '') = COALESCE(replica.browserName, '') AND
    staging.channelId = replica.channelId AND
    COALESCE(staging.city, '') = COALESCE(replica.city, '') AND
    COALESCE(staging.country, '') = COALESCE(replica.country, '') AND
    COALESCE(staging.deviceType, '') = COALESCE(replica.deviceType, '') AND
    DATE(staging.eventDate) = DATE(replica.eventDate) AND
    staging.pageTitle = replica.pageTitle AND
    COALESCE(staging.platformName, '') = COALESCE(replica.platformName, '') AND
    COALESCE(staging.region, '') = COALESCE(replica.region, '') AND
    staging.userId = replica.userId
)

WHEN MATCHED THEN
    UPDATE SET
        replica.clicks = staging.clicks,
        replica.comments = staging.comments,
        replica.ratings = staging.ratings,
        replica.ratingsScore = staging.ratingsScore,
        replica.readTime = staging.readTime,
        replica.sessions = staging.sessions,
        replica.views = staging.views

WHEN NOT MATCHED THEN
    INSERT (
        `assetId`,
        `assetTitle`,
        `browserName`,
        `canonicalUrl`,
        `channelId`,
        `city`,
        `clicks`,
        `comments`,
        `country`,
        `deviceType`,
        `eventDate`,
        `pageTitle`,
        `platformName`,
        `ratings`,
        `ratingsScore`,
        `readTime`,
        `region`,
        `sessions`,
        `userId`,
        `views`
    )
    VALUES (
        staging.assetId,
        staging.assetTitle,
        staging.browserName,
        staging.canonicalUrl,
        staging.channelId,
        staging.city,
        staging.clicks,
        staging.comments,
        staging.country,
        staging.deviceType,
        staging.eventDate,
        staging.pageTitle,
        staging.platformName,
        staging.ratings,
        staging.ratingsScore,
        staging.readTime,
        staging.region,
        staging.sessions,
        staging.userId,
        staging.views
    )
"""

In [ ]:
# response = run_bigquery_query(sql=blogs_daily_merge_sql)

# print(f"Blogs Daily Merge completed with status {response.state}")

#### Custom Assets

In [ ]:
custom_assets_daily_merge_sql = f"""
MERGE INTO
    `{WORKSPACE_ID}.customassetdaily` AS replica
USING
    (
        WITH CustomAssetEvent AS (
            SELECT
                Event.channelId,
                Event.eventDate,
                Event.eventId,
                Event.sessionId,
                TO_HEX(
                    SHA256(
                        CONCAT(
                            Event.assetId,
                            COALESCE(category.value, 'default'),
                            Event.channelId
                        )
                    )
                ) AS assetPrimaryKey,
                formEnabled.value AS formEnabled
            FROM
                `{WORKSPACE_ID}.event` AS Event
            LEFT JOIN `{WORKSPACE_ID}.eventproperty` AS category ON (
                DATE(category.eventDate, '{WORKSPACE_TIME_ZONE_ID}') = '{EVENT_DATE}' AND
                category.id = Event.id AND
                category.name = 'category'
            )
            LEFT JOIN `{WORKSPACE_ID}.eventproperty` AS formEnabled ON (
                DATE(formEnabled.eventDate, '{WORKSPACE_TIME_ZONE_ID}') = '{EVENT_DATE}' AND
                formEnabled.id = Event.id AND
                formEnabled.name = 'formEnabled'
            )
            WHERE
                Event.applicationId = 'Custom' AND
                Event.assetId IS NOT NULL AND
                DATE(Event.eventDate, '{WORKSPACE_TIME_ZONE_ID}') = '{EVENT_DATE}'
        ),
        CustomAssetFinalizedEvent AS (
            SELECT
                CustomAssetEvent.assetPrimaryKey,
                CustomAssetEvent.eventDate,
                CustomAssetEvent.eventId,
                CustomAssetEvent.formEnabled,
                CustomAssetEvent.sessionId
            FROM
                CustomAssetEvent
            INNER JOIN `{WORKSPACE_ID}.session` Session ON
                CustomAssetEvent.sessionId = Session.id
            WHERE
                DATE(Session.sessionStart, '{WORKSPACE_TIME_ZONE_ID}') = '{EVENT_DATE}'
        ),
        Metrics AS (
            SELECT
                assetPrimaryKey,
                channelId,
                COUNTIF(eventId = 'assetClicked') AS clicks,
                COUNTIF(eventId = 'assetDownloaded') AS downloads,
                TIMESTAMP_TRUNC(eventDate, HOUR) AS normalizedEventDate,
                COUNT(DISTINCT(sessionId)) AS sessions,
                COUNTIF(eventId = 'assetSubmitted') AS submissions,
                COUNTIF(eventId = 'assetViewed') AS views
            FROM
                CustomAssetEvent
            GROUP BY
                assetPrimaryKey,
                channelId,
                normalizedEventDate
        ),
        Abandoments AS (
            SELECT
                GREATEST(
                    0,
                    COUNTIF(eventId = 'assetViewed' AND formEnabled = 'true') -
                    COUNTIF(eventId = 'assetSubmitted')
                ) AS abandonments,
                assetPrimaryKey,
                TIMESTAMP_TRUNC(eventDate, HOUR) AS normalizedEventDate
            FROM
                CustomAssetFinalizedEvent
            GROUP BY
                assetPrimaryKey,
                normalizedEventDate
        ),
        ReadTime AS (
            SELECT
                assetPrimaryKey,
                TIMESTAMP_TRUNC(maxEventDate, HOUR) AS normalizedEventDate,
                SUM(readtime) AS readTime
            FROM
                (
                    SELECT
                        assetPrimaryKey,
                        MAX(CASE WHEN eventId != 'assetViewed' THEN eventDate END) AS maxEventDate,
                        UNIX_SECONDS(MAX(CASE WHEN eventId != 'assetViewed' THEN eventDate END)) - UNIX_SECONDS(MIN(eventDate)) AS readTime,
                        sessionId
                    FROM
                        CustomAssetFinalizedEvent
                    GROUP BY
                        assetPrimaryKey,
                        sessionId
                ) AS TMP
            WHERE
                maxEventDate IS NOT NULL
            GROUP BY
                assetprimarykey,
                normalizedEventDate
        ),
        SubmissionTime AS (
            SELECT
                assetPrimaryKey,
                TIMESTAMP_TRUNC(minSubmissionDate, HOUR) AS normalizedEventDate,
                SUM(submissionTime) AS submissionsTime
            FROM
                (
                    SELECT
                        assetprimarykey,
                        MIN(CASE WHEN eventId = 'assetSubmitted' THEN eventDate END) AS minSubmissionDate,
                        sessionid,
                        UNIX_SECONDS(MAX(CASE WHEN eventId = 'assetSubmitted' THEN eventDate END)) - UNIX_SECONDS(MIN(CASE WHEN eventId = 'assetViewed' THEN eventDate END)) AS submissionTime
                    FROM
                        CustomAssetFinalizedEvent
                    GROUP BY
                        assetPrimaryKey,
                        sessionId
                ) AS TMP
            WHERE
                minSubmissionDate IS NOT NULL
            GROUP BY
                assetPrimaryKey,
                normalizedEventDate
        ),
        CustomAssetHourly AS (
            SELECT
                COALESCE(abandonments.abandonments, 0) AS abandonments,
                metrics.assetPrimaryKey,
                metrics.channelId,
                metrics.clicks,
                metrics.downloads,
                metrics.normalizedEventDate AS eventDate,
                metrics.submissions,
                metrics.views,
                metrics.sessions,
                COALESCE (readTime.readTime, 0) * 1000 AS readTime,
                COALESCE (submissionTime.submissionsTime, 0) * 1000 AS submissionsTime
            FROM
                Metrics metrics
            LEFT JOIN Abandoments abandonments ON (
                metrics.assetPrimaryKey = abandonments.assetprimarykey AND
                metrics.normalizedEventDate = abandonments.normalizedEventDate
            )
            LEFT JOIN ReadTime readTime ON (
                metrics.assetPrimaryKey = readTime.assetPrimaryKey AND
                metrics.normalizedEventDate = readTime.normalizedEventDate
            )
            LEFT JOIN SubmissionTime submissionTime ON (
                metrics.assetPrimaryKey = submissionTime.assetPrimaryKey AND
                metrics.normalizedEventDate = submissionTime.normalizedEventDate
            )
        )
        SELECT
            sum(abandonments) as abandonments,
            assetPrimaryKey,
            channelId,
            sum(clicks) as clicks,
            sum(downloads) as downloads,
            TIMESTAMP_TRUNC(eventDate, DAY, '{WORKSPACE_TIME_ZONE_ID}') AS eventDate,
            sum(readTime) as readTime,
            sum(sessions) as sessions,
            sum(submissions) as submissions,
            sum(submissionsTime) as submissionsTime,
            sum(views) as views
        FROM
            CustomAssetHourly
        WHERE
            DATE(eventDate, '{WORKSPACE_TIME_ZONE_ID}') = '{EVENT_DATE}'
        GROUP BY
            assetPrimaryKey, channelId, eventDate
    ) AS staging
ON (
    DATE(replica.eventDate, '{WORKSPACE_TIME_ZONE_ID}') = '{EVENT_DATE}' AND
    staging.assetPrimaryKey = replica.assetPrimaryKey AND
    staging.channelId = replica.channelId AND
    DATE(staging.eventDate) = DATE(replica.eventDate)
)

WHEN MATCHED THEN
    UPDATE SET
        replica.abandonments = staging.abandonments,
        replica.clicks = staging.clicks,
        replica.downloads = staging.downloads,
        replica.readTime = staging.readTime,
        replica.sessions = staging.sessions,
        replica.submissions = staging.submissions,
        replica.submissionsTime = staging.submissionsTime,
        replica.views = staging.views

WHEN NOT MATCHED THEN
    INSERT (
        `abandonments`,
        `assetPrimaryKey`,
        `channelId`,
        `clicks`,
        `downloads`,
        `eventDate`,
        `readTime`,
        `sessions`,
        `submissions`,
        `submissionsTime`,
        `views`
    )
    VALUES (
        staging.abandonments,
        staging.assetPrimaryKey,
        staging.channelId,
        staging.clicks,
        staging.downloads,
        staging.eventDate,
        staging.readTime,
        staging.sessions,
        staging.submissions,
        staging.submissionsTime,
        staging.views
    )
"""

In [ ]:
# response = run_bigquery_query(sql=custom_assets_daily_merge_sql)

# print(f"Custom Assets Daily Merge completed with status {response.state}")

#### Documents

In [ ]:
documents_daily_merge_sql = f"""
MERGE INTO
    `{WORKSPACE_ID}.documentlibrarydaily` AS replica
USING
    (
        WITH
            CommentEvent AS (
                SELECT
                    Event.assetId,
                    Event.canonicalUrl,
                    Event.channelId,
                    Event.eventDate,
                    Event.title,
                    Event.userId
                FROM
                    `{WORKSPACE_ID}.event` AS Event
                LEFT JOIN `{WORKSPACE_ID}.eventproperty` AS className ON (
                    DATE(className.eventDate, '{WORKSPACE_TIME_ZONE_ID}') = '{EVENT_DATE}' AND
                    className.id = Event.id AND
                    className.name = 'className' AND
                    className.value = 'com.liferay.document.library.kernel.model.DLFileEntry'
                )
                WHERE
                    Event.applicationId = 'Comment' AND
                    Event.assetId IS NOT NULL AND
                    Event.assetTitle IS NOT NULL AND
                    Event.canonicalUrl IS NOT NULL AND
                    DATE(Event.eventDate, '{WORKSPACE_TIME_ZONE_ID}') = '{EVENT_DATE}' AND
                    Event.eventId = 'posted' AND
                    Event.title IS NOT NULL
            ),
            DocumentEvent AS (
                SELECT
                    Event.assetId,
                    Event.assetTitle,
                    Event.browserName,
                    Event.canonicalUrl,
                    Event.channelId,
                    Event.city,
                    Event.country,
                    Event.deviceType,
                    Event.eventDate,
                    Event.eventId,
                    Event.platformName,
                    Event.region,
                    Event.title,
                    Event.userId
                FROM
                    `{WORKSPACE_ID}.event` AS Event
                LEFT JOIN `{WORKSPACE_ID}.eventproperty` AS className ON (
                    DATE(className.eventDate, '{WORKSPACE_TIME_ZONE_ID}') = '{EVENT_DATE}' AND
                    className.id = Event.id AND
                    className.name = 'className' AND
                    className.value = 'com.liferay.document.library.kernel.model.DLFileEntry'
                )
                WHERE
                    Event.applicationId = 'Document' AND
                    Event.assetId IS NOT NULL AND
                    Event.assetTitle IS NOT NULL AND
                    Event.canonicalUrl IS NOT NULL AND
                    Event.eventId IN ('documentDownloaded', 'documentPreviewed') AND
                    DATE(Event.eventDate, '{WORKSPACE_TIME_ZONE_ID}') = '{EVENT_DATE}' AND
                    Event.title IS NOT NULL
            ),
            DocumentComments AS (
                SELECT
                    assetId,
                    canonicalUrl,
                    channelId,
                    SUM(1) AS comments,
                    TIMESTAMP_TRUNC(eventDate, HOUR) AS normalizedEventDate,
                    title AS pageTitle,
                    userId
                FROM
                    CommentEvent
                GROUP BY
                    assetId, canonicalUrl, channelId, normalizedEventDate, title, userId
            ),
            DocumentDownloadAndPreviews AS (
                SELECT
                    assetId,
                    assetTitle,
                    browserName,
                    canonicalUrl,
                    channelId,
                    city,
                    COUNTIF(eventId = 'documentDownloaded') AS downloads,
                    country,
                    deviceType,
                    TIMESTAMP_TRUNC(eventDate, HOUR) AS normalizedEventDate,
                    platformName,
                    COUNTIF(eventId = 'documentPreviewed') AS previews,
                    region,
                    title AS pageTitle,
                    userId
                FROM
                    DocumentEvent
                GROUP BY
                    assetId, assetTitle, browserName, canonicalUrl, channelId, city,
                    country, deviceType, normalizedEventDate, platformName,
                    region, title, userId
            ),
            RatingsEvent AS (
                SELECT
                    Event.assetId,
                    Event.canonicalUrl,
                    Event.channelId,
                    Event.eventDate,
                    Event.title,
                    CAST(score.value AS FLOAT64) AS score,
                    Event.userId
                FROM
                    `{WORKSPACE_ID}.event` AS Event
                LEFT JOIN `{WORKSPACE_ID}.eventproperty` AS className ON (
                    DATE(className.eventDate, '{WORKSPACE_TIME_ZONE_ID}') = '{EVENT_DATE}' AND
                    className.id = Event.id AND
                    className.name = 'className' AND
                    className.value = 'com.liferay.document.library.kernel.model.DLFileEntry'
                )
                LEFT JOIN `{WORKSPACE_ID}.eventproperty` AS ratingType ON (
                    DATE(ratingType.eventDate, '{WORKSPACE_TIME_ZONE_ID}') = '{EVENT_DATE}' AND
                    ratingType.id = Event.id AND
                    ratingType.name = 'ratingType' AND
                    ratingtype.value = 'stars'
                )
                LEFT JOIN `{WORKSPACE_ID}.eventproperty` AS score ON (
                    DATE(score.eventDate, '{WORKSPACE_TIME_ZONE_ID}') = '{EVENT_DATE}' AND
                    score.id = Event.id AND
                    score.name = 'score'
                )
                WHERE
                    Event.applicationId = 'Ratings' AND
                    Event.assetId IS NOT NULL AND
                    Event.canonicalUrl IS NOT NULL AND
                    DATE(Event.eventDate, '{WORKSPACE_TIME_ZONE_ID}') = '{EVENT_DATE}' AND
                    Event.eventId = 'VOTE' AND
                    Event.title IS NOT NULL
            ),
            DocumentRatings AS (
                SELECT
                    assetId,
                    canonicalUrl,
                    channelId,
                    TIMESTAMP_TRUNC(eventDate, HOUR) AS normalizedEventDate,
                    title AS pageTitle,
                    SUM(1) AS ratings,
                    score AS ratingsScore,
                    userId
                FROM
                    RatingsEvent AS RatingsEvent1
                WHERE
                    RatingsEvent1.eventDate = (
                        SELECT
                            MAX(RatingsEvent2.eventDate)
                        FROM
                            RatingsEvent RatingsEvent2
                        WHERE
                            RatingsEvent1.assetId = RatingsEvent2.assetId AND
                            RatingsEvent1.userid = RatingsEvent2.userid
                    ) AND score >= 0
                GROUP BY
                    assetId, canonicalUrl, channelId, normalizedEventDate, score,
                    title, userId
            ),
            DocumentLibraryHourly AS (
                SELECT
                    DocumentDownloadAndPreviews.assetId,
                    DocumentDownloadAndPreviews.assetTitle,
                    DocumentDownloadAndPreviews.browserName,
                    DocumentDownloadAndPreviews.canonicalUrl,
                    DocumentDownloadAndPreviews.channelId,
                    DocumentDownloadAndPreviews.city,
                    DocumentComments.comments,
                    DocumentDownloadAndPreviews.country,
                    DocumentDownloadAndPreviews.deviceType,
                    DocumentDownloadAndPreviews.downloads,
                    DocumentDownloadAndPreviews.normalizedEventDate AS eventDate,
                    DocumentDownloadAndPreviews.pageTitle,
                    DocumentDownloadAndPreviews.platformName,
                    DocumentDownloadAndPreviews.previews,
                    DocumentRatings.ratings,
                    DocumentRatings.ratingsScore,
                    DocumentDownloadAndPreviews.region,
                    DocumentDownloadAndPreviews.userId
                FROM
                    DocumentDownloadAndPreviews
                LEFT JOIN DocumentRatings ON (
                    DocumentDownloadAndPreviews.assetId = DocumentRatings.assetId AND
                    DocumentDownloadAndPreviews.canonicalUrl = DocumentRatings.canonicalUrl AND
                    DocumentDownloadAndPreviews.channelId = DocumentRatings.channelId AND
                    DocumentDownloadAndPreviews.normalizedEventDate = DocumentRatings.normalizedEventDate AND
                    DocumentDownloadAndPreviews.pageTitle = DocumentRatings.pageTitle AND
                    DocumentDownloadAndPreviews.userId = DocumentRatings.userId
                )
                LEFT JOIN DocumentComments ON (
                    DocumentDownloadAndPreviews.assetId = DocumentComments.assetId AND
                    DocumentDownloadAndPreviews.canonicalUrl = DocumentComments.canonicalUrl AND
                    DocumentDownloadAndPreviews.channelId = DocumentComments.channelId AND
                    DocumentDownloadAndPreviews.normalizedEventDate = DocumentComments.normalizedEventDate AND
                    DocumentDownloadAndPreviews.pageTitle = DocumentComments.pageTitle AND
                    DocumentDownloadAndPreviews.userId = DocumentComments.userId
                )
            )
        SELECT
            assetId,
            assetTitle,
            browserName,
            canonicalUrl,
            channelId,
            city,
            SUM(comments) AS comments,
            country,
            deviceType,
            SUM(downloads) AS downloads,
            TIMESTAMP_TRUNC(eventDate, DAY, '{WORKSPACE_TIME_ZONE_ID}') AS eventDate,
            pageTitle,
            platformName,
            SUM(previews) AS previews,
            SUM(ratings) AS ratings,
            SUM(ratingsScore) AS ratingsScore,
            region,
            userId
        FROM
            DocumentLibraryHourly
        WHERE
            DATE(eventDate, '{WORKSPACE_TIME_ZONE_ID}') = '{EVENT_DATE}'
        GROUP BY
            assetId, assetTitle, browserName, canonicalUrl, channelId, city,
            country, deviceType, eventDate, pageTitle, platformName, region,
            userId
    ) AS staging
ON (
    DATE(replica.eventDate, '{WORKSPACE_TIME_ZONE_ID}') = '{EVENT_DATE}' AND
    staging.assetId = replica.assetId AND
    staging.assetTitle = replica.assetTitle AND
    COALESCE(staging.browserName, '') = COALESCE(replica.browserName, '') AND
    staging.channelId = replica.channelId AND
    COALESCE(staging.city, '') = COALESCE(replica.city, '') AND
    COALESCE(staging.country, '') = COALESCE(replica.country, '') AND
    COALESCE(staging.deviceType, '') = COALESCE(replica.deviceType, '') AND
    DATE(staging.eventDate) = DATE(replica.eventDate) AND
    staging.pageTitle = replica.pageTitle AND
    COALESCE(staging.platformName, '') = COALESCE(replica.platformName, '') AND
    COALESCE(staging.region, '') = COALESCE(replica.region, '') AND
    staging.userId = replica.userId
)

WHEN MATCHED THEN
    UPDATE SET
        replica.comments = staging.comments,
        replica.downloads = staging.downloads,
        replica.previews = staging.previews,
        replica.ratings = staging.ratings,
        replica.ratingsScore = staging.ratingsScore

WHEN NOT MATCHED THEN
    INSERT (
        `assetId`,
        `assetTitle`,
        `browserName`,
        `canonicalUrl`,
        `channelId`,
        `city`,
        `comments`,
        `country`,
        `deviceType`,
        `downloads`,
        `eventDate`,
        `pageTitle`,
        `platformName`,
        `previews`,
        `ratings`,
        `ratingsScore`,
        `region`,
        `userId`
    )
    VALUES (
        staging.assetId,
        staging.assetTitle,
        staging.browserName,
        staging.canonicalUrl,
        staging.channelId,
        staging.city,
        staging.comments,
        staging.country,
        staging.deviceType,
        staging.downloads,
        staging.eventDate,
        staging.pageTitle,
        staging.platformName,
        staging.previews,
        staging.ratings,
        staging.ratingsScore,
        staging.region,
        staging.userId
    )
"""

In [ ]:
# response = run_bigquery_query(sql=documents_daily_merge_sql)

# print(f"Documents Daily Merge completed with status {response.state}")

#### Forms

In [ ]:
forms_daily_merge_sql = f"""
MERGE INTO
    `{WORKSPACE_ID}.formdaily` AS replica
USING
    (
        WITH
            FormEvent AS (
                SELECT
                    Event.assetId,
                    Event.assetTitle,
                    Event.browserName,
                    Event.canonicalUrl,
                    Event.channelId,
                    Event.city,
                    Event.country,
                    Event.deviceType,
                    Event.eventDate,
                    Event.eventId,
                    Event.platformName,
                    Event.region,
                    Event.sessionId,
                    Event.title,
                    Event.userId
                FROM
                    `{WORKSPACE_ID}.event` AS Event
                WHERE
                    Event.applicationId = 'Form' AND
                    Event.assetId IS NOT NULL AND
                    Event.canonicalUrl IS NOT NULL AND
                    DATE(Event.eventDate, '{WORKSPACE_TIME_ZONE_ID}') = '{EVENT_DATE}' AND
                    Event.eventId IN ('formSubmitted', 'formViewed') AND
                    Event.title IS NOT NULL
            ),
            FormSubmissionTimes AS (
                SELECT
                    assetId,
                    browserName,
                    canonicalUrl,
                    channelId,
                    city,
                    country,
                    deviceType,
                    TIMESTAMP_TRUNC(eventDate, HOUR) AS normalizedEventDate,
                    platformName,
                    region,
                    title AS pageTitle,
                    userId ,
                    SUM(UNIX_SECONDS(eventDate) - UNIX_SECONDS(previousFormViewedEventDate)) * 1000 submissionsTime
                FROM (
                    SELECT
                        FormEvent.assetId,
                        FormEvent.browserName,
                        FormEvent.canonicalUrl,
                        FormEvent.channelId,
                        FormEvent.city,
                        FormEvent.country,
                        FormEvent.deviceType,
                        FormEvent.eventDate,
                        FormEvent.eventId,
                        FormEvent.platformName,
                        FormEvent.region,
                        FormEvent.title,
                        FormEvent.userId,
                        MAX(CASE WHEN eventId = 'formViewed' THEN eventDate END)
                        OVER (
                            PARTITION BY
                                assetId, channelId, canonicalUrl, sessionId, title
                            ORDER BY
                                eventDate ASC
                            ROWS UNBOUNDED PRECEDING
                        ) AS previousFormViewedEventDate
                    FROM
                        FormEvent
                ) AS TMP
                WHERE
                    eventId = 'formSubmitted'
                GROUP BY
                    assetId, browserName, canonicalUrl, channelId, city,
                    country, deviceType, normalizedEventDate, platformName,
                    region, title, userId
            ),
            FormHourly AS (
                SELECT
                    GREATEST(
                        0,
                        COUNTIF(eventId = 'formViewed' AND Session.id IS NOT NULL) -
                        COUNTIF(eventId = 'formSubmitted' AND Session.id IS NOT NULL)
                    ) AS abandonments,
                    FormEvent.assetId,
                    COALESCE(MAX(FormEvent.assetTitle), '') AS assetTitle,
                    FormEvent.browserName,
                    FormEvent.canonicalUrl,
                    CAST(FormEvent.channelId AS INT64) AS channelId,
                    FormEvent.city,
                    FormEvent.country,
                    FormEvent.deviceType,
                    TIMESTAMP_TRUNC(eventDate, HOUR) AS eventDate,
                    COUNTIF(eventId = 'formViewed' AND Session.id IS NOT NULL
                    ) AS finalizedFormViews,
                    FormEvent.platformName,
                    FormEvent.region,
                    FormEvent.title AS pageTitle,
                    COUNTIF(eventId = 'formSubmitted') AS submissions,
                    MAX(FormSubmissionTimes.submissionsTime) AS submissionsTime,
                    FormEvent.userId,
                    COUNTIF(eventId = 'formViewed') AS views
                FROM
                    FormEvent
                LEFT JOIN FormSubmissionTimes ON (
                    FormEvent.assetId = FormSubmissionTimes.assetId AND
                    FormEvent.browserName = FormSubmissionTimes.browserName AND
                    FormEvent.canonicalUrl = FormSubmissionTimes.canonicalUrl AND
                    FormEvent.channelId = FormSubmissionTimes.channelId AND
                    FormEvent.city = FormSubmissionTimes.city AND
                    FormEvent.country = FormSubmissionTimes.country AND
                    FormEvent.deviceType = FormSubmissionTimes.deviceType AND
                    TIMESTAMP_TRUNC(eventDate, HOUR) = FormSubmissionTimes.normalizedEventDate AND
                    FormEvent.platformName = FormSubmissionTimes.platformName AND
                    FormEvent.region = FormSubmissionTimes.region AND
                    FormEvent.title = FormSubmissionTimes.pageTitle AND
                    FormEvent.userId = FormSubmissionTimes.userId)
                LEFT JOIN `{WORKSPACE_ID}.session` AS Session ON
                    FormEvent.sessionId = Session.id AND
                    DATE(Session.sessionStart, '{WORKSPACE_TIME_ZONE_ID}') = '{EVENT_DATE}'
                GROUP BY
                    assetId, browserName, canonicalUrl, channelId, city, country, deviceType,
                    eventDate, platformName, region, title, userId
            )
        SELECT
            SUM(abandonments) AS abandonments,
            assetId,
            assetTitle,
            browserName,
            canonicalUrl,
            channelId,
            city,
            country,
            deviceType,
            TIMESTAMP_TRUNC(eventDate, DAY, '{WORKSPACE_TIME_ZONE_ID}') AS eventDate,
            SUM(finalizedFormViews) AS finalizedFormViews,
            pageTitle,
            platformName,
            region,
            SUM(submissions) AS submissions,
            SUM(submissionsTime) AS submissionsTime,
            userId,
            SUM(views) AS views
        FROM
            FormHourly
        WHERE
            DATE(eventDate, '{WORKSPACE_TIME_ZONE_ID}') = '{EVENT_DATE}'
        GROUP BY
            assetId, assetTitle, browserName, canonicalUrl, channelId, city,
            country, deviceType, eventDate, pageTitle, platformName, region,
            userId
    ) AS staging
ON (
    DATE(replica.eventDate, '{WORKSPACE_TIME_ZONE_ID}') = '{EVENT_DATE}' AND
    staging.assetId = replica.assetId AND
    staging.assetTitle = replica.assetTitle AND
    COALESCE(staging.browserName, '') = COALESCE(replica.browserName, '') AND
    staging.channelId = replica.channelId AND
    COALESCE(staging.city, '') = COALESCE(replica.city, '') AND
    COALESCE(staging.country, '') = COALESCE(replica.country, '') AND
    COALESCE(staging.deviceType, '') = COALESCE(replica.deviceType, '') AND
    DATE(staging.eventDate) = DATE(replica.eventDate) AND
    staging.pageTitle = replica.pageTitle AND
    COALESCE(staging.platformName, '') = COALESCE(replica.platformName, '') AND
    COALESCE(staging.region, '') = COALESCE(replica.region, '') AND
    staging.userId = replica.userId
)

WHEN MATCHED THEN
    UPDATE SET
        replica.abandonments = staging.abandonments,
        replica.finalizedFormViews = staging.finalizedFormViews,
        replica.submissions = staging.submissions,
        replica.submissionsTime = staging.submissionsTime,
        replica.views = staging.views
WHEN NOT MATCHED THEN
    INSERT (
        `abandonments`,
        `assetId`,
        `assetTitle`,
        `browserName`,
        `canonicalUrl`,
        `channelId`,
        `city`,
        `country`,
        `deviceType`,
        `eventDate`,
        `finalizedFormViews`,
        `pageTitle`,
        `platformName`,
        `region`,
        `submissions`,
        `submissionsTime`,
        `userId`,
        `views`
    )
    VALUES (
        staging.abandonments,
        staging.assetId,
        staging.assetTitle,
        staging.browserName,
        staging.canonicalUrl,
        staging.channelId,
        staging.city,
        staging.country,
        staging.deviceType,
        staging.eventDate,
        staging.finalizedFormViews,
        staging.pageTitle,
        staging.platformName,
        staging.region,
        staging.submissions,
        staging.submissionsTime,
        staging.userId,
        staging.views
    )
"""

In [ ]:
# response = run_bigquery_query(sql=forms_daily_merge_sql)

# print(f"Forms Daily Merge completed with status {response.state}")

#### Page

In [ ]:
page_merge_daily_sql =f"""
MERGE INTO
    `{WORKSPACE_ID}.pagedaily` AS replica
USING
    (
    WITH PageEvent AS (
            SELECT
                applicationId,
                COALESCE(browserName, '') AS browserName,
                canonicalUrl,
                channelId,
                COALESCE(city, '') AS city,
                COALESCE(country, '') AS country,
                COALESCE(description, '') AS description,
                COALESCE(deviceType, '') AS deviceType,
                eventDate,
                eventId,
                experimentId,
                COALESCE(platformName, '') AS platformName,
                referrer,
                COALESCE(region, '') AS region,
                sessionId,
                title,
                url,
                userId,
                variantId
            FROM
                `{WORKSPACE_ID}.event` AS Event
            WHERE
                DATE(eventDate, '{WORKSPACE_TIME_ZONE_ID}') = '{EVENT_DATE}'
        ),
        PageFinalizedEvent AS (
                SELECT
                PageEvent.applicationId,
                PageEvent.browserName,
                PageEvent.canonicalUrl,
                PageEvent.channelId,
                PageEvent.city,
                PageEvent.country,
                PageEvent.deviceType,
                PageEvent.eventDate,
                PageEvent.eventId,
                PageEvent.platformName,
                PageEvent.region,
                PageEvent.sessionId,
                PageEvent.title,
                PageEvent.userId
                FROM
                    PageEvent
            INNER JOIN
                `{WORKSPACE_ID}.session` AS Session ON
                        PageEvent.sessionId = Session.id
                WHERE
                DATE(Session.sessionStart, '{WORKSPACE_TIME_ZONE_ID}') = '{EVENT_DATE}'
        ),
        PageBounces AS (
            SELECT
                PageFinalizedEvent.channelId,
                COUNT(*) AS count,
                COUNTIF(
                    PageFinalizedEvent.applicationId = 'Page' AND
                    PageFinalizedEvent.eventId = 'pageViewed'
                ) AS pageViews,
                PageFinalizedEvent.sessionId,
                PageFinalizedEvent.userId
            FROM
                PageFinalizedEvent
            WHERE
                PageFinalizedEvent.eventId NOT IN (
                    'blogViewed', 'documentPreviewed', 'formViewed', 'pageLoaded',
                    'pageUnloaded', 'webContentViewed'
                    )
            GROUP BY
                channelId, sessionId, userId
        ),
        PageEntrances AS (
            SELECT
                browserName,
                canonicalUrl,
                channelId,
                city,
                country,
                deviceType,
                rank AS entrances,
                TIMESTAMP_TRUNC(eventDate, HOUR) AS normalizedEventDate,
                platformName,
                region,
                sessionId,
                title,
                userId
            FROM (
                SELECT
                    PageFinalizedEvent.browserName,
                    PageFinalizedEvent.canonicalUrl,
                    PageFinalizedEvent.channelId,
                    PageFinalizedEvent.city,
                    PageFinalizedEvent.country,
                    PageFinalizedEvent.deviceType,
                    PageFinalizedEvent.eventDate,
                    PageFinalizedEvent.platformName,
                    ROW_NUMBER() OVER (
                        PARTITION BY
                            PageFinalizedEvent.channelId,
                            PageFinalizedEvent.sessionId,
                            PageFinalizedEvent.userId
                        ORDER BY
                            PageFinalizedEvent.eventDate ASC
                    ) AS rank,
                    PageFinalizedEvent.region,
                    PageFinalizedEvent.sessionId,
                    PageFinalizedEvent.title,
                    PageFinalizedEvent.userId
                FROM
                    PageFinalizedEvent
            ) AS EventEntrance
            WHERE
                rank = 1
        ),
        PageExits AS (
            SELECT
                browserName,
                canonicalUrl,
                channelId,
                city,
                country,
                deviceType,
                rank AS exits,
                TIMESTAMP_TRUNC(eventDate, HOUR) AS normalizedEventDate,
                platformName,
                region,
                sessionId,
                title,
                userId
            FROM (
                SELECT
                    PageFinalizedEvent.browserName,
                    PageFinalizedEvent.canonicalUrl,
                    PageFinalizedEvent.channelId,
                    PageFinalizedEvent.city,
                    PageFinalizedEvent.country,
                    PageFinalizedEvent.deviceType,
                    PageFinalizedEvent.eventDate,
                    PageFinalizedEvent.platformName,
                    ROW_NUMBER() OVER (
                        PARTITION BY
                            PageFinalizedEvent.channelId,
                            PageFinalizedEvent.sessionId,
                            PageFinalizedEvent.userId
                        ORDER BY
                            PageFinalizedEvent.eventDate DESC
                    ) AS rank,
                    PageFinalizedEvent.region,
                    PageFinalizedEvent.sessionId,
                    PageFinalizedEvent.title,
                    PageFinalizedEvent.userId
                FROM
                    PageFinalizedEvent
            ) AS EventExit
            WHERE
                rank = 1
        ),
        PageTimeOnPages AS (
            SELECT
                browserName,
                canonicalUrl,
                channelId,
                city,
                country,
                deviceType,
                TIMESTAMP_TRUNC(eventDate, HOUR) AS normalizedEventDate,
                platformName,
                region,
                sessionId,
                SUM(TIMESTAMP_DIFF(nextTime, eventDate, MILLISECOND)) AS timeOnPage,
                title,
                userId
            FROM (
                SELECT
                    PageFinalizedEvent.browserName,
                    PageFinalizedEvent.canonicalUrl,
                    PageFinalizedEvent.channelId,
                    PageFinalizedEvent.city,
                    PageFinalizedEvent.country,
                    PageFinalizedEvent.deviceType,
                    PageFinalizedEvent.eventDate,
                    LEAD(
                            PageFinalizedEvent.eventDate
                    ) OVER (
                        PARTITION BY
                            PageFinalizedEvent.channelId,
                            PageFinalizedEvent.sessionId,
                            PageFinalizedEvent.userId
                        ORDER BY
                            PageFinalizedEvent.eventDate
                    ) AS nextTime,
                    PageFinalizedEvent.platformName,
                    PageFinalizedEvent.region,
                    PageFinalizedEvent.sessionId,
                    PageFinalizedEvent.title,
                    PageFinalizedEvent.userId
                FROM
                    PageFinalizedEvent
            ) AS EventTimeOnPage
            GROUP BY
                browserName, canonicalUrl, channelId, city, country, deviceType,
                normalizedEventDate, platformName, region, sessionId, title, userId
        ),
        PageViews AS (
            SELECT
                browserName,
                canonicalUrl,
                channelId,
                city,
                country,
                COUNTIF(eventId = 'ctaClicked') ctaClicks,
                MAX(description) AS description,
                deviceType,
                COUNTIF(eventId = 'pageViewed' AND referrer = '') AS directAccess,
                ANY_VALUE(experimentId) AS experimentId,
                COUNTIF(eventId = 'pageViewed' AND referrer != '') AS indirectAccess,
                TIMESTAMP_TRUNC(eventDate, HOUR) AS normalizedEventDate,
                platformName,
                COUNTIF(eventId = 'pageRead') reads,
                region,
                sessionId,
                title,
                userId,
                ANY_VALUE(variantId) AS variantId,
                COUNTIF(eventId = 'pageViewed') AS views
            FROM
                PageEvent
            WHERE
                applicationId = 'Page' AND
                eventId IN ('ctaClicked', 'pageRead', 'pageViewed')
            GROUP BY
                browserName, canonicalUrl, channelId, city, country, deviceType,
                normalizedEventDate, platformName, region, sessionId, title, userId
        ),
        PageHourly AS (
                SELECT
                CASE
                    WHEN
                        PageBounces.count > 2 OR PageBounces.pageViews > 1
                    THEN
                        0
                    ELSE
                        1
                END AS bounce,
                PageViews.browserName,
                PageViews.canonicalUrl,
                PageViews.channelId,
                PageViews.city,
                PageViews.country,
                PageViews.ctaClicks,
                PageViews.description,
                PageViews.deviceType,
                PageViews.directAccess,
                PageEntrances.entrances,
                PageViews.normalizedEventDate AS eventDate,
                PageExits.exits,
                PageViews.experimentId,
                PageViews.indirectAccess,
                PageViews.platformName,
                PageViews.reads,
                PageViews.region,
                PageTimeOnPages.sessionId,
                PageTimeOnPages.timeOnPage,
                PageViews.title,
                PageViews.userId,
                PageViews.variantId,
                PageViews.views
            FROM
                PageViews
            LEFT JOIN PageBounces ON (
                PageViews.channelId = PageBounces.channelId AND
                COALESCE(PageViews.sessionId, '') = COALESCE(PageBounces.sessionId, '') AND
                PageViews.userId = PageBounces.userId
            )
            LEFT JOIN PageEntrances ON (
                PageViews.browserName = PageEntrances.browserName AND
                PageViews.canonicalUrl = PageEntrances.canonicalUrl AND
                PageViews.channelId = PageEntrances.channelId AND
                PageViews.city = PageEntrances.city AND
                PageViews.country = PageEntrances.country AND
                PageViews.deviceType = PageEntrances.deviceType AND
                PageViews.normalizedEventDate = PageEntrances.normalizedEventDate AND
                PageViews.platformName = PageEntrances.platformName AND
                PageViews.region = PageEntrances.region AND
                COALESCE(PageViews.sessionId, '') = COALESCE(PageEntrances.sessionId, '') AND
                PageViews.title = PageEntrances.title AND
                PageViews.userId = PageEntrances.userId
            )
            LEFT JOIN PageExits ON (
                PageViews.browserName = PageExits.browserName AND
                PageViews.canonicalUrl = PageExits.canonicalUrl AND
                PageViews.channelId = PageExits.channelId AND
                PageViews.city = PageExits.city AND
                PageViews.country = PageExits.country AND
                PageViews.deviceType = PageExits.deviceType AND
                PageViews.normalizedEventDate = PageExits.normalizedEventDate AND
                PageViews.platformName = PageExits.platformName AND
                PageViews.region = PageExits.region AND
                COALESCE(PageViews.sessionId, '') = COALESCE(PageExits.sessionId, '') AND
                PageViews.title = PageExits.title AND
                PageViews.userId = PageExits.userId
            )
            LEFT JOIN PageTimeOnPages ON (
                PageViews.browserName = PageTimeOnPages.browserName AND
                PageViews.canonicalUrl = PageTimeOnPages.canonicalUrl AND
                PageViews.channelId = PageTimeOnPages.channelId AND
                PageViews.city = PageTimeOnPages.city AND
                PageViews.country = PageTimeOnPages.country AND
                PageViews.deviceType = PageTimeOnPages.deviceType AND
                PageViews.normalizedEventDate = PageTimeOnPages.normalizedEventDate AND
                PageViews.platformName = PageTimeOnPages.platformName AND
                PageViews.region = PageTimeOnPages.region AND
                COALESCE(PageViews.sessionId, '') = COALESCE(PageTimeOnPages.sessionId, '') AND
                PageViews.title = PageTimeOnPages.title AND
                PageViews.userId = PageTimeOnPages.userId
            )
        )
        SELECT
            SUM(bounce) AS bounce,
            browserName,
            canonicalUrl,
            channelId,
            city,
            country,
            SUM(ctaClicks) AS ctaClicks,
            description,
            deviceType,
            SUM(directAccess) AS directAccess,
            SUM(entrances) AS entrances,
            TIMESTAMP_TRUNC(eventDate, DAY, '{WORKSPACE_TIME_ZONE_ID}') AS eventDate,
            SUM(exits) exits,
            ANY_VALUE(experimentId) AS experimentId,
            SUM(indirectAccess) AS indirectAccess,
            platformName,
            SUM(reads) AS reads,
            region,
            sessionId,
            SUM(timeOnPage) AS timeOnPage,
            title,
            userId,
            ANY_VALUE(variantId) AS variantId,
            SUM(views) AS views
        FROM
            PageHourly
        WHERE
            DATE(eventDate, '{WORKSPACE_TIME_ZONE_ID}') = '{EVENT_DATE}' AND
            sessionId IS NOT NULL
        GROUP BY
            browserName, canonicalUrl, channelId, city, country, description,
            deviceType, eventDate, platformName, region, sessionId, title,
            userId
    ) AS staging
ON (
    DATE(replica.eventDate, '{WORKSPACE_TIME_ZONE_ID}') = '{EVENT_DATE}' AND
    COALESCE(staging.browserName, '') = COALESCE(replica.browserName, '') AND
    staging.canonicalUrl = replica.canonicalUrl AND
    staging.channelId = replica.channelId AND
    COALESCE(staging.city, '') = COALESCE(replica.city, '') AND
    COALESCE(staging.country, '') = COALESCE(replica.country, '') AND
    COALESCE(staging.deviceType, '') = COALESCE(replica.deviceType, '') AND
    DATE(staging.eventDate) = DATE(replica.eventDate) AND
    COALESCE(staging.platformName, '') = COALESCE(replica.platformName, '') AND
    COALESCE(staging.region, '') = COALESCE(replica.region, '') AND
    staging.sessionId = replica.sessionId AND
    staging.title = replica.title AND
    staging.userId = replica.userId
)

WHEN MATCHED THEN
    UPDATE SET
        replica.bounce = staging.bounce,
        replica.ctaClicks = staging.ctaClicks,
        replica.directAccess = staging.directAccess,
        replica.entrances = staging.entrances,
        replica.exits = staging.exits,
        replica.indirectAccess = staging.indirectAccess,
        replica.reads = staging.reads,
        replica.timeOnPage = staging.timeOnPage,
        replica.views = staging.views
        
WHEN NOT MATCHED THEN
    INSERT (
        `bounce`,
        `browserName`,
        `canonicalUrl`,
        `channelId`,
        `city`,
        `country`,
        `ctaClicks`,
        `description`,
        `deviceType`,
        `directAccess`,
        `entrances`,
        `eventDate`,
        `exits`,
        `experimentId`,
        `indirectAccess`,
        `platformName`,
        `reads`,
        `region`,
        `sessionId`,
        `timeOnPage`,
        `title`,
        `userId`,
        `variantId`,
        `views`
    )
    VALUES (
        staging.bounce,
        staging.browserName,
        staging.canonicalUrl,
        staging.channelId,
        staging.city,
        staging.country,
        staging.ctaClicks,
        staging.description,
        staging.deviceType,
        staging.directAccess,
        staging.entrances,
        staging.eventDate,
        staging.exits,
        staging.experimentId,
        staging.indirectAccess,
        staging.platformName,
        staging.reads,
        staging.region,
        staging.sessionId,
        staging.timeOnPage,
        staging.title,
        staging.userId,
        staging.variantId,
        staging.views
    )
"""

In [ ]:
# response = run_bigquery_query(sql=page_merge_daily_sql)

# print(f"Page Daily Merge completed with status {response.state}")

#### Journal

In [ ]:
journal_daily_merge_sql = f"""
MERGE INTO
    `{WORKSPACE_ID}.journaldaily` AS replica
USING
    (
        WITH
            WebContentEvent AS (
                SELECT
                    Event.assetId,
                    Event.assetTitle,
                    Event.browserName,
                    Event.canonicalUrl,
                    Event.channelId,
                    Event.city,
                    Event.country,
                    Event.deviceType,
                    Event.eventDate,
                    Event.platformName,
                    Event.region,
                    Event.title,
                    Event.userId
                FROM
                    `{WORKSPACE_ID}.event` AS Event
                WHERE
                    Event.applicationId = 'WebContent' AND
                    Event.assetId IS NOT NULL AND
                    Event.assetTitle IS NOT NULL AND
                    Event.canonicalUrl IS NOT NULL AND
                    Event.channelId IS NOT NULL AND
                    DATE(Event.eventDate, '{WORKSPACE_TIME_ZONE_ID}') = '{EVENT_DATE}' AND
                    Event.eventId = 'webContentViewed' AND
                    Event.title IS NOT NULL
            ),
            JournalHourly AS (
                SELECT
                    assetId,
                    assetTitle,
                    browserName,
                    canonicalUrl,
                    channelId,
                    city,
                    country,
                    deviceType,
                    TIMESTAMP_TRUNC(eventDate, HOUR) AS eventDate,
                    platformName,
                    region,
                    title AS pageTitle,
                    userId,
                    SUM(1) AS views
                FROM
                    WebContentEvent
                GROUP BY
                    assetId, assetTitle, browserName, canonicalUrl, channelId, city,
                    country, TIMESTAMP_TRUNC(eventDate, HOUR), deviceType, platformName,
                    region, title, userId
            )
        SELECT
            assetId,
            assetTitle,
            browserName,
            canonicalUrl,
            channelId,
            city,
            country,
            deviceType,
            TIMESTAMP_TRUNC(eventDate, DAY, '{WORKSPACE_TIME_ZONE_ID}') AS eventDate,
            pageTitle,
            platformName,
            region,
            userId,
            SUM(views) AS views
        FROM
            JournalHourly
        WHERE
            DATE(eventDate, '{WORKSPACE_TIME_ZONE_ID}') = '{EVENT_DATE}'
        GROUP BY
            assetId, assetTitle, browserName, canonicalUrl, channelId, city,
            country, deviceType, eventDate, pageTitle, platformName, region,
            userId
    ) AS staging
ON (
    DATE(replica.eventDate, '{WORKSPACE_TIME_ZONE_ID}') = '{EVENT_DATE}' AND
    staging.assetId = replica.assetId AND
    staging.assetTitle = replica.assetTitle AND
    COALESCE(staging.browserName, '') = COALESCE(replica.browserName, '') AND
    staging.channelId = replica.channelId AND
    COALESCE(staging.city, '') = COALESCE(replica.city, '') AND
    COALESCE(staging.country, '') = COALESCE(replica.country, '') AND
    COALESCE(staging.deviceType, '') = COALESCE(replica.deviceType, '') AND
    DATE(staging.eventDate) = DATE(replica.eventDate) AND
    staging.pageTitle = replica.pageTitle AND
    COALESCE(staging.platformName, '') = COALESCE(replica.platformName, '') AND
    COALESCE(staging.region, '') = COALESCE(replica.region, '') AND
    staging.userId = replica.userId
)

WHEN MATCHED THEN
    UPDATE SET
        replica.views = staging.views

WHEN NOT MATCHED THEN
    INSERT (
        `assetId`,
        `assetTitle`,
        `browserName`,
        `canonicalUrl`,
        `channelId`,
        `city`,
        `country`,
        `deviceType`,
        `eventDate`,
        `pageTitle`,
        `platformName`,
        `region`,
        `userId`,
        `views`
    )
    VALUES (
        staging.assetId,
        staging.assetTitle,
        staging.browserName,
        staging.canonicalUrl,
        staging.channelId,
        staging.city,
        staging.country,
        staging.deviceType,
        staging.eventDate,
        staging.pageTitle,
        staging.platformName,
        staging.region,
        staging.userId,
        staging.views
    )
"""

In [ ]:
# response = run_bigquery_query(sql=journal_daily_merge_sql)

# print(f"Journal Daily Merge completed with status {response.state}")

#### Identity Activity Summary

In [ ]:
identity_activity_summary_daily_merge_sql = f"""
MERGE INTO
    `{WORKSPACE_ID}.identityactivitysummary` AS replica
USING
    (
        SELECT
            COUNT(*) AS activitiesCount,
            Event.channelId,
            Event.dataSourceId,
            Event.eventId,
            MIN(Event.eventDate) AS firstActivityDate,
            Event.userId AS identityId,
            MAX(Identity.individualId) AS individualId,
            MAX(Event.eventDate) AS lastActivityDate
        FROM
            `{WORKSPACE_ID}.event` AS Event
        LEFT JOIN `{WORKSPACE_ID}.identity` AS Identity ON (
            Event.userId = Identity.id
        )
        WHERE
            DATE(Event.eventDate, '{WORKSPACE_TIME_ZONE_ID}') = '{EVENT_DATE}'
        GROUP BY
            Event.channelId,
            Event.dataSourceId,
            Event.eventId,
            Event.userId
    ) AS staging
ON (
    staging.channelId = replica.channelId AND
    staging.dataSourceId = replica.dataSourceId AND
    staging.eventId = replica.eventId AND
    staging.identityId = replica.identityId
)

WHEN MATCHED THEN
    UPDATE SET
        replica.activitiesCount = replica.activitiesCount + staging.activitiesCount,
        replica.individualId = CASE WHEN replica.individualId IS NOT NULL THEN replica.individualId ELSE (CASE WHEN staging.individualId IS NOT NULL THEN staging.individualId END) END,
        replica.lastActivityDate = GREATEST(staging.lastActivityDate, replica.lastActivityDate)
WHEN NOT MATCHED THEN
    INSERT (
        `activitiesCount`,
        `channelId`,
        `dataSourceId`,
        `eventId`,
        `firstActivityDate`,
        `identityId`,
        `individualId`,
        `lastActivityDate`
    )
    VALUES (
        staging.activitiesCount,
        staging.channelId,
        staging.dataSourceId,
        staging.eventId,
        staging.firstActivityDate,
        staging.identityId,
        staging.individualId,
        staging.lastActivityDate
    )
"""

In [ ]:
# response = run_bigquery_query(sql=identity_activity_summary_daily_merge_sql)

# print(f"Identity Activity Summary Daily Merge completed with status {response.state}")